In [0]:
asset_raw_text_df = (
    spark.read
        .text("s3://enterprise-lakehouse-data/bronze/asset_snapshots/")
)

asset_raw_text_df.show(5, truncate=False)


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|value                                                                                                                                                                                                                                                                                                                             |ingestion_date|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

DEFINE SCHEMA FOR THIS SOURCE


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

asset_payload_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("plant_id", StringType(), True),
    StructField("asset_type", StringType(), True),
    StructField("install_date", StringType(), True),   # keep as string for now
    StructField("capacity_mw", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("last_updated", StringType(), True)
])

asset_envelope_schema = StructType([
    StructField("payload", asset_payload_schema, True),
    StructField("kafka_topic", StringType(), True),
    StructField("kafka_partition", IntegerType(), True),
    StructField("kafka_offset", IntegerType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])


PARSE JSON SAFELY



In [0]:
from pyspark.sql.functions import from_json, col

parsed_asset_df = (
    asset_raw_text_df
        .withColumn(
            "json_data",
            from_json(col("value"), asset_envelope_schema)
        )
)


EXPAND STRUCTURE

In [0]:
asset_structured_df = (
    parsed_asset_df
        .select(
            col("json_data.payload.*"),
            col("json_data.kafka_topic"),
            col("json_data.kafka_partition"),
            col("json_data.kafka_offset"),
            col("json_data.ingestion_timestamp"),
            col("ingestion_date")
        )
)


In [0]:
asset_structured_df.printSchema()
asset_structured_df.show(5, truncate=False)


root
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)

+--------+--------+----------+------------+-----------+-------+------------+-------------------+---------------+------------+---------------------------+--------------+
|asset_id|plant_id|asset_type|install_date|capacity_mw|status |last_updated|kafka_topic        |kafka_partition|kafka_offset|ingestion_timestamp        |ingestion_date|
+--------+--------+----------+------------+-----------+-------+------------+-------------------+---------------+------------+---------

In [0]:
asset_structured_df.show()

+--------+--------+----------+------------+-----------+--------+------------+-------------------+---------------+------------+--------------------+--------------+
|asset_id|plant_id|asset_type|install_date|capacity_mw|  status|last_updated|        kafka_topic|kafka_partition|kafka_offset| ingestion_timestamp|ingestion_date|
+--------+--------+----------+------------+-----------+--------+------------+-------------------+---------------+------------+--------------------+--------------+
|AST-1158|    NULL|      PUMP|  2025-10-28|      200.0| RETIRED|  2025-06-17|asset_snapshots_raw|              0|       15100|2026-01-07T13:49:...|    2026-01-07|
|AST-1174| PLANT-4|      PUMP|  2025-08-24|       50.0| RETIRED|  2025-12-17|asset_snapshots_raw|              0|       15101|2026-01-07T13:49:...|    2026-01-07|
|AST-1200|PLANT-10|      PUMP|  2025-08-13|      -50.0|  ACTIVE|  2026-02-07|asset_snapshots_raw|              0|       15102|2026-01-07T13:49:...|    2026-01-07|
|AST-1021| PLANT-5|   

In [0]:
asset_structured_df.select("asset_type").distinct().show()

+----------+
|asset_type|
+----------+
|      PUMP|
| GENERATOR|
|   TURBINE|
|     SOLAR|
+----------+



CAST DATE/TIMESTAMP (CONTROLLED)

In [0]:
from pyspark.sql.functions import to_date, to_timestamp

asset_typed_df = (
    asset_structured_df
        .withColumn("install_date_dt", to_date("install_date"))
        .withColumn("last_updated_dt", to_date("last_updated"))
        .withColumn("ingestion_ts", to_timestamp("ingestion_timestamp"))
)


DEFINE VALIDATION CONDITIONS (BOOLEAN FLAGS)

In [0]:
from pyspark.sql.functions import col, current_date

validated_flags_df = (
    asset_typed_df
        # ---------- NULL CHECKS ----------
        .withColumn("f_null_asset_id", col("asset_id").isNull())
        .withColumn("f_null_asset_type", col("asset_type").isNull())
        .withColumn("f_null_status", col("status").isNull())
        .withColumn("f_null_ingestion_ts", col("ingestion_ts").isNull())

        # ---------- DOMAIN CHECKS ----------
        .withColumn(
            "f_invalid_asset_type",
            ~col("asset_type").isin("PUMP", "GENERATOR", "TURBINE", "SOLAR")
        )
        .withColumn(
            "f_invalid_status",
            ~col("status").isin("ACTIVE", "INACTIVE", "MAINTENANCE", "RETIRED")
        )

        # ---------- RANGE CHECKS ----------
        .withColumn("f_invalid_capacity", col("capacity_mw") <= 0)

        # ---------- TEMPORAL CHECKS ----------
        .withColumn("f_future_last_updated", col("last_updated_dt") > current_date())
        .withColumn("f_install_after_ingestion", col("install_date_dt") > col("ingestion_date"))
)


BUILD failure_reason AS ARRAY

In [0]:
from pyspark.sql.functions import array, when, lit, expr
with_failure_array_df = (
    validated_flags_df
        .withColumn(
            "failure_reason",
            expr("""
                filter(
                    array(
                        CASE WHEN f_null_asset_id THEN 'NULL_ASSET_ID' END,
                        CASE WHEN f_null_asset_type THEN 'NULL_ASSET_TYPE' END,
                        CASE WHEN f_null_status THEN 'NULL_STATUS' END,
                        CASE WHEN f_null_ingestion_ts THEN 'NULL_INGESTION_TIMESTAMP' END,

                        CASE WHEN f_invalid_asset_type THEN 'INVALID_ASSET_TYPE' END,
                        CASE WHEN f_invalid_status THEN 'INVALID_STATUS' END,

                        CASE WHEN f_invalid_capacity THEN 'NEGATIVE_OR_ZERO_CAPACITY' END,

                        CASE WHEN f_future_last_updated THEN 'FUTURE_LAST_UPDATED' END,
                        CASE WHEN f_install_after_ingestion THEN 'INSTALL_DATE_AFTER_INGESTION' END
                    ),
                    x -> x IS NOT NULL
                )
            """)
        )
)


ADD QUARANTINE METADATA

In [0]:
from pyspark.sql.functions import current_timestamp
quarantine_ready_df = (
    with_failure_array_df
        .withColumn("dataset", lit("asset_snapshots"))
        .withColumn("processed_at", current_timestamp())
)


SPLIT SILVER vs QUARANTINE

In [0]:
silver_asset_df = quarantine_ready_df.filter(
    expr("size(failure_reason) = 0")
)
quarantine_asset_df = quarantine_ready_df.filter(
    expr("size(failure_reason) > 0")
)


In [0]:
quarantine_asset_df.select(
    "asset_id",
    "asset_type",
    "capacity_mw",
    "failure_reason"
).show(20, truncate=False)


+--------+----------+-----------+---------------------------------------------------------+
|asset_id|asset_type|capacity_mw|failure_reason                                           |
+--------+----------+-----------+---------------------------------------------------------+
|AST-1200|PUMP      |-50.0      |[NEGATIVE_OR_ZERO_CAPACITY, FUTURE_LAST_UPDATED]         |
|AST-1029|PUMP      |100.0      |[FUTURE_LAST_UPDATED]                                    |
|AST-1129|SOLAR     |100.0      |[FUTURE_LAST_UPDATED]                                    |
|AST-1035|SOLAR     |200.0      |[INSTALL_DATE_AFTER_INGESTION]                           |
|AST-1032|GENERATOR |-100.0     |[NEGATIVE_OR_ZERO_CAPACITY, INSTALL_DATE_AFTER_INGESTION]|
|AST-1071|PUMP      |-200.0     |[NEGATIVE_OR_ZERO_CAPACITY, INSTALL_DATE_AFTER_INGESTION]|
|AST-1096|TURBINE   |100.0      |[INSTALL_DATE_AFTER_INGESTION]                           |
|AST-1088|PUMP      |100.0      |[INSTALL_DATE_AFTER_INGESTION]                 

COMPARE

In [0]:
print(asset_structured_df.count())
print(silver_asset_df.count())
print(quarantine_asset_df.count())


20100
3503
16597


In [0]:
asset_structured_df.count()==silver_asset_df.count()+quarantine_asset_df.count()

True

STORING QUARANTINE TO S3

In [0]:
quarantine_write_df = quarantine_asset_df.select(
    "dataset",
    "asset_id",
    "plant_id",
    "asset_type",
    "install_date",
    "capacity_mw",
    "status",
    "last_updated",

    "kafka_topic",
    "kafka_partition",
    "kafka_offset",

    "ingestion_timestamp",
    "ingestion_date",

    "failure_reason",
    "processed_at"
)


In [0]:
quarantine_write_df.printSchema()

root
 |-- dataset: string (nullable = false)
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- failure_reason: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- processed_at: timestamp (nullable = false)



In [0]:
(
    quarantine_write_df
        .write
        .mode("overwrite")                 # idempotent per partition
        .partitionBy("ingestion_date")
        .parquet(
            "s3://enterprise-lakehouse-data/quarantine/asset_snapshots/"
        )
)


In [0]:
from pyspark.sql.functions import sha2, col
silver_hashed_df = (
    silver_asset_df
        .withColumn(
            "asset_hash",
            sha2(col("asset_id"), 256)
        )
)


In [0]:
silver_hashed_df.select("asset_id", "asset_hash").show(5, truncate=False)


+--------+----------------------------------------------------------------+
|asset_id|asset_hash                                                      |
+--------+----------------------------------------------------------------+
|AST-1158|4e15deddda6d28ebe17dba9f5f32d636d63eb3b29d96211bb70c2b7b11e0fb23|
|AST-1174|3291e7fc5d33af85e881d31c69a3011953e91e433f3e9cf254ac0f52468ffa8b|
|AST-1021|c5a37a93fa1acf3a0a81f86ca2fa518b36b336cd75bb190977c47c7181d0f429|
|AST-1190|1fd8b773a993117d90fe2b40ddc1f701d2cdb7494dff6baeb8c960dbc55c91e7|
|AST-1181|ac0477fdbeb4ccfeceefcf31707d4e40f91f76f917d6ef63b8775bbe032fea74|
+--------+----------------------------------------------------------------+
only showing top 5 rows


In [0]:
silver_hashed_df.printSchema()

root
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- install_date_dt: date (nullable = true)
 |-- last_updated_dt: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- f_null_asset_id: boolean (nullable = false)
 |-- f_null_asset_type: boolean (nullable = false)
 |-- f_null_status: boolean (nullable = false)
 |-- f_null_ingestion_ts: boolean (nullable = false)
 |-- f_invalid_asset_type: boolean (nullable = true)
 |-- f_invalid_status: boolean (nullable = true)
 |-- f_invalid_capacity: boolean (

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
dedup_window = (
    Window
        .partitionBy("asset_hash")
        .orderBy(col("ingestion_ts").desc())
)
silver_dedup_df = (
    silver_hashed_df
        .withColumn("row_num", row_number().over(dedup_window))
        .filter(col("row_num") == 1)
        .drop("row_num")
)


In [0]:
silver_dedup_df.select(
    "asset_id",
    "asset_hash",
    "ingestion_ts"
).show(10, truncate=False)


+--------+----------------------------------------------------------------+--------------------------+
|asset_id|asset_hash                                                      |ingestion_ts              |
+--------+----------------------------------------------------------------+--------------------------+
|AST-1020|000fb7a779aa2c49430bf38de242031975d131c2081740db2eab7748fb38b29d|2026-01-07 13:49:10.118876|
|AST-1077|01a62459051f02b93cced3af66713721732cf28eff95a5be028f27b37ad70c6d|2026-01-07 13:49:09.937822|
|AST-1063|026a3e542735d9e3a9132efdf69f1f6025ba1b31fd3bbd2ed83510f65387fde8|2026-01-07 13:49:10.11834 |
|AST-1137|0388f620b7306749d51191fa97dc56e79bfc602b00334b78e2f9c6e9908b6814|2026-01-07 13:49:10.088283|
|AST-1033|04106bbcc45beb7542a1ed19bcabe490c3756ecb97d7a2faad211643286f9f0c|2026-01-07 13:49:10.118445|
|AST-1061|0583e9ed78c50938b02f07672a86c69866c076613c8c6165f2f47240d4775268|2026-01-07 13:49:10.117956|
|AST-1040|07c1e53282e99a796cc6db556f5ccf9eca5826ec45a7d7cf17a1a3a532ce775

In [0]:
from pyspark.sql.functions import datediff
silver_lateness_df = (
    silver_dedup_df
        .withColumn(
            "lateness_days",
            datediff(col("ingestion_date"), col("last_updated_dt"))
        )
)


In [0]:
from pyspark.sql.functions import when, lit
silver_lateness_df = (
    silver_lateness_df
        .withColumn(
            "is_late_record",
            when(col("lateness_days") > 1, lit(True)).otherwise(lit(False))
        )
)


In [0]:
silver_final_df = silver_lateness_df.select(
    "asset_id",
    "plant_id",
    "asset_type",
    "capacity_mw",
    "status",
    "install_date_dt",
    "last_updated_dt",

    "asset_hash",
    "ingestion_date",

    "lateness_days",
    "is_late_record"
)


In [0]:
silver_final_df.printSchema()
silver_final_df.show(5, truncate=False)


root
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- install_date_dt: date (nullable = true)
 |-- last_updated_dt: date (nullable = true)
 |-- asset_hash: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- lateness_days: integer (nullable = true)
 |-- is_late_record: boolean (nullable = false)

+--------+--------+----------+-----------+--------+---------------+---------------+----------------------------------------------------------------+--------------+-------------+--------------+
|asset_id|plant_id|asset_type|capacity_mw|status  |install_date_dt|last_updated_dt|asset_hash                                                      |ingestion_date|lateness_days|is_late_record|
+--------+--------+----------+-----------+--------+---------------+---------------+----------------------------------------------------

In [0]:
silver_final_df.show()

+--------+--------+----------+-----------+--------+---------------+---------------+--------------------+--------------+-------------+--------------+
|asset_id|plant_id|asset_type|capacity_mw|  status|install_date_dt|last_updated_dt|          asset_hash|ingestion_date|lateness_days|is_late_record|
+--------+--------+----------+-----------+--------+---------------+---------------+--------------------+--------------+-------------+--------------+
|AST-1020|    NULL| GENERATOR|      100.0|INACTIVE|     2025-11-17|     2025-12-22|000fb7a779aa2c494...|    2026-01-07|           16|          true|
|AST-1077| PLANT-9| GENERATOR|      200.0|  ACTIVE|     2025-07-02|     2025-04-21|01a62459051f02b93...|    2026-01-07|          261|          true|
|AST-1063| PLANT-7|     SOLAR|      100.0|INACTIVE|     2025-06-05|     2025-10-08|026a3e542735d9e3a...|    2026-01-07|           91|          true|
|AST-1137| PLANT-1|      PUMP|       50.0| RETIRED|     2025-09-12|     2026-01-01|0388f620b7306749d...|  

In [0]:
silver_final_df.count()

201

REFERENCE_DATA

In [0]:
reference_raw_df = (
    spark.read
        .text("s3://enterprise-lakehouse-data/bronze/reference_data/")
)

print("Raw count:", reference_raw_df.count())
reference_raw_df.show(5, truncate=False)


Raw count: 5050
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|value                                                                                                                                                                                                         |ingestion_date|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|{"payload": {"asset_type": "SOLAR", "risk": "LOW"}, "kafka_topic": "reference_data_raw", "kafka_partition": 0, "kafka_offset": 50, "ingestion_timestamp": "2026-01-07T13:49:30.951350Z"}                      |2026-01-07    |
|{"payload": {"asset_type": "SOLAR", "risk": "LOW"}, "kafka_topic": "reference_data_raw"

DEFINE SCHEMA

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
reference_payload_schema = StructType([
    StructField("asset_type", StringType(), True),
    StructField("risk", StringType(), True),
    StructField("cycle_days", IntegerType(), True)
])

reference_envelope_schema = StructType([
    StructField("payload", reference_payload_schema, True),
    StructField("kafka_topic", StringType(), True),
    StructField("kafka_partition", IntegerType(), True),
    StructField("kafka_offset", IntegerType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])


PARSE JSON

In [0]:
from pyspark.sql.functions import from_json, col
reference_parsed_df = (
    reference_raw_df
        .withColumn(
            "json",
            from_json(col("value"), reference_envelope_schema)
        )
        .select(
            col("json.payload.*"),
            col("json.ingestion_timestamp"),
            col("ingestion_date")
        )
)


TYPE CASTING

In [0]:
from pyspark.sql.functions import to_timestamp
reference_typed_df = (
    reference_parsed_df
        .withColumn("ingestion_ts", to_timestamp("ingestion_timestamp"))
)


In [0]:
reference_typed_df.show(10, truncate=False)


+----------+------+----------+---------------------------+--------------+--------------------------+
|asset_type|risk  |cycle_days|ingestion_timestamp        |ingestion_date|ingestion_ts              |
+----------+------+----------+---------------------------+--------------+--------------------------+
|SOLAR     |LOW   |NULL      |2026-01-07T13:49:30.951350Z|2026-01-07    |2026-01-07 13:49:30.95135 |
|SOLAR     |LOW   |NULL      |2026-01-07T13:49:30.951366Z|2026-01-07    |2026-01-07 13:49:30.951366|
|SOLAR     |LOW   |NULL      |2026-01-07T13:49:30.951369Z|2026-01-07    |2026-01-07 13:49:30.951369|
|SOLAR     |LOW   |NULL      |2026-01-07T13:49:30.951372Z|2026-01-07    |2026-01-07 13:49:30.951372|
|PUMP      |MEDIUM|NULL      |2026-01-07T13:49:30.951374Z|2026-01-07    |2026-01-07 13:49:30.951374|
|PUMP      |MEDIUM|NULL      |2026-01-07T13:49:30.951376Z|2026-01-07    |2026-01-07 13:49:30.951376|
|PUMP      |MEDIUM|NULL      |2026-01-07T13:49:30.951378Z|2026-01-07    |2026-01-07 13:49:3

VALIDATION (REFERENCE SAFE)

In [0]:
from pyspark.sql.functions import col
reference_flags_df = (
    reference_typed_df
        .withColumn("f_null_asset_type", col("asset_type").isNull())
        .withColumn("f_null_risk", col("risk").isNull())
        .withColumn("f_null_ingestion_ts", col("ingestion_ts").isNull())
)


failure_reason ARRAY

In [0]:
from pyspark.sql.functions import expr
reference_with_failure_df = (
    reference_flags_df
        .withColumn(
            "failure_reason",
            expr("""
                filter(
                    array(
                        CASE WHEN f_null_asset_type THEN 'NULL_ASSET_TYPE' END,
                        CASE WHEN f_null_risk THEN 'NULL_RISK' END,
                        CASE WHEN f_null_ingestion_ts THEN 'NULL_INGESTION_TIMESTAMP' END
                    ),
                    x -> x IS NOT NULL
                )
            """)
        )
)


SPLIT SILVER vs QUARANTINE

In [0]:
reference_silver_df = reference_with_failure_df.filter(expr("size(failure_reason) = 0"))
reference_quarantine_df = reference_with_failure_df.filter(expr("size(failure_reason) > 0"))


In [0]:
print(reference_silver_df.count())
print(reference_quarantine_df.count())

5050
0


HASHING

In [0]:
from pyspark.sql.functions import sha2
reference_hashed_df = (
    reference_silver_df
        .withColumn("reference_hash", sha2(col("asset_type"), 256))
)


DEDUPLICATION (LATEST ONLY)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
dedup_window = (
    Window
        .partitionBy("asset_type")
        .orderBy(col("ingestion_ts").desc())
)
reference_dedup_df = (
    reference_hashed_df
        .withColumn("row_num", row_number().over(dedup_window))
        .filter(col("row_num") == 1)
        .drop("row_num")
)


In [0]:
reference_silver_final_df = reference_dedup_df.select(
    "asset_type",
    "risk",
    "cycle_days",
    "reference_hash",
    "ingestion_date"
)


In [0]:
reference_silver_final_df.count()

4

In [0]:
(
    reference_silver_final_df
        .write
        .mode("overwrite")
        .partitionBy("ingestion_date")
        .parquet("s3://enterprise-lakehouse-data/silver/reference_data/")
)
